In [1]:
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(batchelor))
suppressPackageStartupMessages(library(argparse))

here::i_am("mapping/run/mnn/mapping_mnn.R")

# Load mapping functions
source(here::here("mapping/run/mnn/mapping_functions_extended.R"))

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/01_Eomes_RNA/code



In [2]:
# I/O
io$path2atlas <- io$atlas.basedir
io$path2query <- io$basedir

# START TEST ##
args = list()
args$atlas_stages <- c("E8.5")
args$query_samples <- opts$samples[1]
args$query_sce <- io$rna.sce
args$query_sce <- paste0(io$basedir,"/processed/SingleCellExperiment.rds")
args$atlas_sce <- io$rna.atlas.sce
args$query_metadata <- paste0(io$basedir,"/results/rna/doublet_detection/sample_metadata_after_doublets.txt.gz")
args$atlas_metadata <- io$rna.atlas.metadata
args$test <- FALSE
args$npcs <- 5
args$n_neighbours <- 25
args$use_marker_genes <- FALSE
args$cosine_normalisation <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/mapping/test")
# END TEST ##

In [3]:
args$query_samples

[1] "SLX-20795_SITTH11_HKTG2DRXY"

In [4]:
################
## Load query ##
################

# Load cell metadata
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$query_samples]
if (isTRUE(args$test)) meta_query <- head(meta_query,n=1000)

# Load SingleCellExperiment
sce_query <- load_SingleCellExperiment(args$query_sce, cells = meta_query$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_query %>% .[cell%in%colnames(sce_query)] %>% setkey(cell) %>% .[colnames(sce_query)]
stopifnot(tmp$cell == colnames(sce_query))
colData(sce_query) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_query),] %>% DataFrame()

################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(args$atlas_metadata) %>%
  .[stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)] 

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=1000)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(args$atlas_sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

# Sanity cehcks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)

In [5]:
#####################
## Define gene set ##
#####################

# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

rownames(sce_atlas) = gene_metadata[match(rownames(sce_atlas), ens_id), symbol]

# Imprinted genes
imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
                       grep('paternally', gene_metadata$description)), symbol]
#Other imprinted genes: 
#- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
#- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes.intersect,invert=T)] # filter out non-informative genes
genes.intersect <- genes.intersect[grep("^Hbb|^Hba",genes.intersect,invert=T)] # test removing Haem genes 
genes.intersect <- genes.intersect[!genes.intersect %in% c(imprint, 'Grb10', 'Nnat')] # remove imprinted genes
genes.intersect <- genes.intersect[!genes.intersect %in% c("Xist", "Tsix")] # remove Xist & Tsix
genes.intersect <- genes.intersect[!genes.intersect=="tomato-td"] # remove tomato itself
genes.intersect <- genes.intersect[!genes.intersect %in% gene_metadata[chr=="chrY",symbol]] # no genes on y-chr 


In [7]:
length(genes.intersect)

[1] 11889

In [8]:
# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [17]:
  # Calculate mean-variance relationship and extract HVGs
  decomp <- modelGeneVar(sce_atlas, block=sce_atlas$sample)
  genes_to_use <- decomp[order(decomp$FDR),] %>% head(n=2500) %>% rownames
genes_to_use

[1] "Mdk"          "Tmsb4x"       "Car2"         "Marcksl1"    
   [5] "Id3"          "Mest"         "Tmsb10"       "Serpinh1"    
   [9] "Phlda2"       "Crabp1"       "Tuba1a"       "Fth1"        
  [13] "Gpx1"         "Myl7"         "Actc1"        "Bex1"        
  [17] "Blvrb"        "Tmem14c"      "Krt18"        "Myl4"        
  [21] "Cdkn1c"       "Acta2"        "Meg3"         "Cited4"      
  [25] "Ttr"          "Crabp2"       "Bex3"         "Mgst3"       
  [29] "Actg1"        "Ifitm1"       "Apom"         "Ptn"         
  [33] "Rbp4"         "Ccnd2"        "Apoa1"        "Apoe"        
  [37] "Krt8"         "Acta1"        "Prdx2"        "Glrx5"       
  [41] "Id2"          "Hoxaas3"      "H19"          "Hmbs"        
  [45] "Cited2"       "Tnnc1"        "Myl2"         "Amn"         
  [49] "Lgals2"       "Car4"         "Slc2a3"       "Tpm1"        
  [53] "Cldn6"        "Cited1"       "Ube2c"        "Bex4"        
  [57] "Gpx3"         "S100g"        "Apoa2"        "Smim1"       
  [61] "Rhox5"        "Dlk1"         "Cd24a"        "Prdx1"       
  [65] "Cryab"        "Spink1"       "Ctsh"         "Afp"         
  [69] "Tnni1"        "Pyy"          "Fbxl22"       "Igfbp2"      
  [73] "Dppa3"        "Cpox"         "H2afv"        "Pgam1"       
  [77] "Malat1"       "Pmp22"        "Ifitm2"       "Cldn5"       
  [81] "Tcf15"        "Hand1"        "Tagln"        "Chchd10"     
  [85] "Sparc"        "Alad"         "Slc2a1"       "Trap1a"      
  [89] "Snca"         "Hebp1"        "Hspa5"        "Fabp5"       
  [93] "Pkm"          "Vim"          "Csrp2"        "Car7"        
  [97] "Hoxb5os"      "Lgals1"       "Fgb"          "Nkx2-9"      
 [101] "Tnni3"        "Meox1"        "Pdia6"        "Cnn2"        
 [105] "Cd63"         "Arg1"         "Fxyd2"        "Epcam"       
 [109] "Cubn"         "Acp5"         "Slc25a5"      "Sh3bgr"      
 [113] "Fgf8"         "Wnt6"         "Folr1"        "Slc25a4"     
 [117] "Emb"          "Fst"          "Tpm4"         "Crip2"       
 [121] "Cyp26a1"      "Etv2"         "Tdo2"         "Cnn3"        
 [125] "Tdh"          "Hspb1"        "Plac1"        "Apob"        
 [129] "Ass1"         "Mt1"          "Cldn7"        "Bhlha9"      
 [133] "Cck"          "Tmem37"       "Plvap"        "Id1"         
 [137] "T"            "Bex2"         "Krt19"        "Msx1"        
 [141] "Cdx4"         "Anxa2"        "Msx3"         "Rrm2"        
 [145] "Cd81"         "Cdx2"         "Hes3"         "Myl9"        
 [149] "Hoxc8"        "Hspb2"        "Fscn1"        "Cnn1"        
 [153] "Myl6"         "Plek"         "Mt2"          "Col1a1"      
 [157] "Slc16a3"      "Npl"          "Rspo3"        "Hes7"        
 [161] "Dll3"         "Pgk1"         "Sfrp5"        "Pdzk1"       
 [165] "Sfn"          "Pnpo"         "Selenop"      "S100a10"     
 [169] "Fmr1nb"       "Ddt"          "Rrad"         "Atpif1"      
 [173] "Actn2"        "Phlda1"       "Sostdc1"      "Fech"        
 [177] "Foxf1"        "Vgll3"        "Ftl1"         "Dqx1"        
 [181] "Pitx1"        "Stmn1"        "Trh"          "S100a1"      
 [185] "Nebl"         "Ctsl"         "Dab2"         "Fam210b"     
 [189] "Arl6ip1"      "Calca"        "Actb"         "Tsen34"      
 [193] "Slc39a8"      "Olfr1369-ps1" "Ube2l6"       "Fgf17"       
 [197] "Ecscr"        "Cd34"         "Ramp2"        "Aqp3"        
 [201] "Lmo2"         "Vamp8"        "Rax"          "Nkx6-2"      
 [205] "Gpc3"         "Spin2c"       "Psmb8"        "Rbp1"        
 [209] "Ccnd3"        "Slc38a5"      "Hoxb9"        "Nexn"        
 [213] "Wisp1"        "H1f0"         "Psip1"        "Krt7"        
 [217] "Prrx2"        "Cald1"        "Tead2"        "En1"         
 [221] "Apela"        "Hdgf"         "Lgmn"         "Wfdc2"       
 [225] "Slc25a37"     "Hapln1"       "Hoxc9"        "Gpx2"        
 [229] "Apoc1"        "Bambi"        "Tceal8"       "Fam162a"     
 [233] "Gngt2"        "H2afy2"       "Dkk4"         "Hoxa9"       
 [237] "Sox17"        "Gp1bb"        "Prdx6"      

In [25]:
library(Seurat)

Attaching SeuratObject


Attaching package: ‘Seurat’


The following object is masked from ‘package:SummarizedExperiment’:

    Assays




In [43]:
seurat_features = VariableFeatures(FindVariableFeatures(as.Seurat(sce_atlas), nfeatures = 2500))

In [42]:
summary(gene_stats.dt[match(VariableFeatures(seurat_features), gene),]$var_pseudobulk)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.010   0.540   1.110   1.435   1.930   8.940 

In [46]:
  # Calculate mean-variance relationship and extract HVGs
  decomp <- modelGeneVar(sce_atlas)
  genes_to_use <- decomp[order(decomp$FDR),] %>% head(n=2500) %>% rownames

In [47]:
summary(gene_stats.dt[match(genes_to_use, gene),]$var_pseudobulk)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.160   0.500   1.072   1.552   8.940 

In [48]:
head(gene_stats.dt[match(genes_to_use, gene),])

ens_id,mean_single_cells,var_single_cells,mean_pseudobulk,var_pseudobulk,gene
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ENSMUSG00000051855,1.70,2.81,7.67,4.27,Mest
ENSMUSG00000079523,4.34,1.71,10.69,0.88,Tmsb10
ENSMUSG00000027562,1.87,2.68,7.55,3.59,Car2
ENSMUSG00000047945,3.98,1.17,10.03,1.01,Marcksl1
ENSMUSG00000072235,2.61,2.23,8.60,2.75,Tuba1a
ENSMUSG00000049775,2.11,2.00,8.49,1.24,Tmsb4x


In [56]:
  decomp <- decomp[decomp$mean > 0.5,]
  decomp$FDR <- p.adjust(decomp$p.value, method = "fdr")
  genes_to_use <- decomp[order(decomp$FDR),] %>% head(n=2500) %>% rownames

In [57]:
head(gene_stats.dt[match(genes_to_use, gene),])

ens_id,mean_single_cells,var_single_cells,mean_pseudobulk,var_pseudobulk,gene
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ENSMUSG00000051855,1.70,2.81,7.67,4.27,Mest
ENSMUSG00000079523,4.34,1.71,10.69,0.88,Tmsb10
ENSMUSG00000027562,1.87,2.68,7.55,3.59,Car2
ENSMUSG00000047945,3.98,1.17,10.03,1.01,Marcksl1
ENSMUSG00000072235,2.61,2.23,8.60,2.75,Tuba1a
ENSMUSG00000049775,2.11,2.00,8.49,1.24,Tmsb4x


In [58]:
summary(gene_stats.dt[match(genes_to_use, gene),]$var_pseudobulk)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0100  0.0600  0.1100  0.3682  0.2300  8.9400 

In [59]:
summary(gene_stats.dt %>% setorder(-var_pseudobulk, na.last = T) %>% head(n=2500) %>% .$var_pseudobulk)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.71    0.94    1.29    1.67    1.99    8.94 

In [34]:
head(gene_stats.dt)

ens_id,mean_single_cells,var_single_cells,mean_pseudobulk,var_pseudobulk,gene
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ENSMUSG00000085696,0.34,1.04,3.80,8.94,Hoxaas3
ENSMUSG00000025902,0.27,0.60,3.19,8.19,Sox17
ENSMUSG00000062327,0.44,0.98,4.63,7.92,T
ENSMUSG00000005892,0.59,1.41,4.18,7.67,Trh
ENSMUSG00000021765,0.62,1.19,4.98,7.36,Fst
ENSMUSG00000025219,0.64,1.22,4.91,7.02,Fgf8


In [15]:

  # Load gene statistics from the atlas
  gene_stats.dt <- fread(paste0(io$atlas.basedir,"/results/gene_statistics/gene_statistics.txt.gz")) %>%
    .[gene%in%genes.intersect]
  genes_to_use <- gene_stats.dt %>% setorder(-var_pseudobulk, na.last = T) %>% head(n=2500) %>% .$gene 
genes_to_use

[1] "Hoxaas3"    "Sox17"      "T"          "Trh"        "Fst"       
   [6] "Fgf8"       "Cdx1"       "Pou5f1"     "Mesp1"      "Cldn6"     
  [11] "Cited4"     "Fgf5"       "Cdx2"       "Ptn"        "Epcam"     
  [16] "Acta2"      "Utf1"       "Crabp2"     "Hand1"      "Cdx4"      
  [21] "Crabp1"     "Hoxb5os"    "Sp5"        "Eomes"      "Fam212a"   
  [26] "Phlda2"     "Mixl1"      "Hoxb8"      "Hoxb2"      "Hoxc8"     
  [31] "Cfc1"       "Foxf1"      "Hoxb1"      "Tal1"       "Lmo2"      
  [36] "Rspo3"      "Cldn7"      "Arg1"       "Hoxa1"      "Hapln1"    
  [41] "Fgf3"       "Cyp26a1"    "Krt19"      "Hes3"       "Cryab"     
  [46] "Snca"       "Gal"        "Krt8"       "Gata6"      "Prrx2"     
  [51] "Lhx1"       "Ifitm1"     "Aplnr"      "Hoxb9"      "Hes7"      
  [56] "Lefty2"     "Krt18"      "Otx2"       "Rhox5"      "Hoxb6"     
  [61] "Crb3"       "Myl7"       "Hoxa7"      "Hhex"       "Smim1"     
  [66] "Pmp22"      "Trap1a"     "Actc1"      "Dlk1"       "Foxb1"     
  [71] "Hoxa9"      "Nkx1-2"     "Meis2"      "Dll3"       "Eras"      
  [76] "Hoxc9"      "Tnni1"      "Apela"      "Plac1"      "Rab25"     
  [81] "Aldh1a2"    "Dkk1"       "Igfbp2"     "Isl1"       "Mest"      
  [86] "Myl4"       "Kdr"        "Gata4"      "Car4"       "Evx1"      
  [91] "Sfrp1"      "Cited1"     "Cdh1"       "Upp1"       "Gsc"       
  [96] "Ppp1r1a"    "Krt7"       "Pdlim4"     "Sfrp5"      "Pim2"      
 [101] "Spink1"     "Calca"      "Gypc"       "Meox1"      "Fgf17"     
 [106] "Sox2"       "Tbx6"       "Cthrc1"     "Hotairm1"   "Bmp4"      
 [111] "Tinagl1"    "Nrp1"       "Hoxb4"      "Evx1os"     "Frzb"      
 [116] "Car3"       "Mgst1"      "Lef1"       "Foxj1"      "Slc7a3"    
 [121] "Igf2"       "Slc39a8"    "Sparc"      "Stmn2"      "Plet1"     
 [126] "Pitx1"      "Etv2"       "Fabp7"      "Cdkn1c"     "Zic3"      
 [131] "Pifo"       "Vim"        "Ldhb"       "Igfbp5"     "Rgs5"      
 [136] "Id2"        "Msx1"       "Dppa5a"     "Bmp2"       "Mef2c"     
 [141] "Pitx2"      "H19"        "Msx2"       "Ccnd2"      "Cpn1"      
 [146] "Hoxd9"      "Gap43"      "Snai1"      "Hoxb3os"    "Pth1r"     
 [151] "Car2"       "Flt1"       "Ina"        "Spin2c"     "Meg3"      
 [156] "Zfp503"     "Ccnd1"      "Cldn5"      "Wfdc2"      "Fli1"      
 [161] "Tcea3"      "Tmod1"      "Tagln"      "Wnt5b"      "Efna1"     
 [166] "Fabp3"      "Sfn"        "Tnnc1"      "Ap1m2"      "Hoxa5"     
 [171] "Hoxc6"      "Smarcd3"    "Cystm1"     "Ecscr"      "Pou3f1"    
 [176] "Ube2l6"     "Hoxb7"      "Hand2"      "Fbn2"       "Irx3"      
 [181] "Mt2"        "Igfbpl1"    "Hoxb5"      "Chchd10"    "Lhx1os"    
 [186] "Epor"       "Sox4"       "Hesx1"      "Cntfr"      "Hoxd1"     
 [191] "Gjb3"       "Sct"        "Shisa2"     "Cdh2"       "Gbx2"      
 [196] "Cnn2"       "Amn"        "Arhgdib"    "Ctsh"       "Slc38a5"   
 [201] "Wnt6"       "Cd59a"      "Fzd2"       "Fgf15"      "Mt1"       
 [206] "Mfap2"      "Prdm6"      "Kdelr3"     "Tfpi"       "Anxa2"     
 [211] "Gadd45g"    "Fgfbp1"     "Osr1"       "Ttr"        "Hoxd4"     
 [216] "Gata2"      "Vrtn"       "Pdgfa"      "Blvrb"      "Sox9"      
 [221] "Folr1"      "Hspb1"      "Tppp3"      "Pdpn"       "Igdcc3"    
 [226] "Basp1"      "Capn6"      "Tmprss2"    "Acp5"       "Dll1"      
 [231] "Grb7"       "Cldn4"      "Hoxa2"      "Asb4"       "Spint2"    
 [236] "Gng11"      "Capsl"      "Nog"        "Cldn9"      "Bcat1"     
 [241] "Col4a2"     "Fmr1nb"     "Sh3bgr"     "Vamp5"      "Epha5"     
 [246] "Epha1"      "Tmem88"     "Perp"       "Rasgrp3"    "Dok4"      
 [251] "Phlda1"     "Kcnk1"      "Gpx2"       "Ramp2"      "H2afy2"    
 [256] "Emb"        "Apom"       "Cck"        "Gjc1"       "Cxcl12"    
 [261] "Irx5"       "Apln"       "Ovol2"      "Wnt3a"      "Col9a1"    
 [266] "Tgfb1"      "Cbfa2t3"    "Efna3"      "Prss8"      "Col4a1"    
 [271] "Twist1"     "Car14"      "Hebp1"      "Bst2"       "Selenop"   
 [276] "Fxyd6"      "Apoe"       "Nexn"    

# MapWrap unpacked

In [48]:
  sce_atlas = sce_atlas
  meta_atlas = meta_atlas
  sce_query = sce_query
  meta_query = meta_query
  genes = genes_to_use
  npcs = args$npcs
  k = args$n_neighbours
  cosineNorm = args$cosine_normalisation
  order = NULL

In [49]:
   
  # Normalisation
  message(sprintf("Normalizing joint dataset using cosineNorm=%s...",cosineNorm))
  sce_all <- joint.normalisation(sce_query, sce_atlas, cosineNorm)
  message("Done\n")

Normalizing joint dataset using cosineNorm=FALSE...

Done




In [50]:
  
  # Feature selection
  if (is.null(genes)) {
    message("Genes not provided. Computing highly variable genes...")
    # hvgs <- getHVGs(sce_all, block=c(meta_atlas$sample, meta_query$sample))
    genes <- getHVGs(sce_all, block=sce_all$block)
    message("Done\n")
  } else {
    message(sprintf("%d Genes provided...",length(genes)))
  }

2500 Genes provided...



In [51]:
  # Dimensionality reduction
  message("Performing PCA...")
  big_pca <- multiBatchPCA(
    sce_all,
    batch = sce_all$block,
    subset.row = genes,
    d = npcs,
    preserve.single = TRUE,
    assay.type = if (cosineNorm) "cosineNorm" else "logcounts"
  )[[1]]
  rownames(big_pca) <- colnames(sce_all) 
  atlas_pca <- big_pca[1:ncol(sce_atlas),]
  query_pca   <- big_pca[-(1:ncol(sce_atlas)),]
  message("Done\n")

Performing PCA...

Done




In [52]:
  
  # Batch effect correction for the atlas
  message("Batch effect correction for the atlas...")  
  order_df        <- meta_atlas[!duplicated(meta_atlas$sample), c("stage", "sample")]
  order_df$ncells <- sapply(order_df$sample, function(x) sum(meta_atlas$sample == x))
  order_df$stage  <- factor(order_df$stage, levels = rev(c("E9.5",
                                       "E9.25",
                                       "E9.0",
                                       "E8.75",
                                       "E8.5",
                                       "E8.25",
                                       "E8.0",
                                       "E7.75",
                                       "E7.5",
                                       "E7.25",
                                       "mixed_gastrulation",
                                       "E7.0",
                                       "E6.75",
                                       "E6.5")))
  order_df       <- order_df[order(order_df$stage, order_df$ncells, decreasing = TRUE),]
  order_df$stage <- as.character(order_df$stage)
  
  set.seed(42)
  pca_atlas_corrected <- doBatchCorrect(counts         = logcounts(sce_atlas[genes,]), 
                                    timepoints      = meta_atlas$stage, 
                                    samples         = meta_atlas$sample, 
                                    timepoint_order = order_df$stage, 
                                    sample_order    = order_df$sample, 
                                    pc_override     = atlas_pca,
                                    npc             = npcs)
  message("Done\n")

Batch effect correction for the atlas...

Loading required package: BiocParallel

Done




In [54]:
  # Mapping query to batch-corrected atlas
  message("MNN mapping...")              
  # correct <- reducedMNN(rbind(pca_atlas_corrected, query_pca), batch = sce_all$block)[["corrected"]]
  correct <- reducedMNN(rbind(pca_atlas_corrected, query_pca),
                      # batch=c(rep("ATLAS", dim(meta_atlas)[1]), meta_query$sample),
                      batch = as.character(sce_all$block),
                      merge.order = order)$corrected
  pca_atlas_corrected <- correct[1:nrow(atlas_pca),]
  pca_query_corrected   <- correct[-(1:nrow(atlas_pca)),]

MNN mapping...



In [56]:
head(pca_query_corrected)
head(pca_atlas_corrected)

SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1,-14.987037,2.413490,24.6406411,-3.032402,5.767520
SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1,-7.940671,7.905278,-14.2722171,-16.816031,10.581002
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1,-8.726419,8.758685,-0.9571918,-8.901286,-1.698947
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1,-13.366618,3.451749,23.3918650,-5.954716,0.584606
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTGATAGTA-1,-15.784547,1.576234,23.8164706,-1.795425,6.121257
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTTAAGACA-1,-8.859923,9.763747,2.7037113,-5.096928,-4.358171


cell_361,-22.18608,-5.858202,-7.477659,13.22464068,-8.494032
cell_362,-12.44580,6.015637,2.600329,-1.15762130,2.156080
cell_363,-10.66507,8.345092,7.358897,-3.35566261,3.066436
cell_364,-10.77508,6.928303,9.191929,-4.99766121,-4.773238
cell_365,-12.98190,5.618987,1.879723,0.01229901,2.966236
cell_366,-10.58148,6.606713,6.264774,-6.22517346,-5.805861


In [59]:
meta_atlas$celltype = meta_atlas$celltype_extended_atlas

In [60]:
  mapping <- get_meta(pca_atlas = pca_atlas_corrected,
                      meta_atlas = meta_atlas,
                      pca_query = pca_query_corrected,
                      meta_query = meta_query,
                      k = k)
  message("Done\n")

Done




In [61]:

  

  


  



  
  # Mapping scores
  message("Computing mapping scores...") 
  out <- list()
  for (i in seq(from = 1, to = k)) {
    out$closest.cells[[i]]     <- sapply(mapping, function(x) x$cells.mapped[i])
    out$celltypes.mapped[[i]]  <- sapply(mapping, function(x) x$celltypes.mapped[i])
    out$cellstages.mapped[[i]] <- sapply(mapping, function(x) x$stages.mapped[i])
  }  
  multinomial.prob <- getMappingScore(out)
  message("Done\n")
  
  # Prepare output
  message("Writing output...") 
  out$pca_atlas_corrected <- pca_atlas_corrected
  out$pca_query_corrected <- pca_query_corrected
  ct <- sapply(mapping, function(x) x$celltype.mapped); is.na(ct) <- lengths(ct) == 0
  st <- sapply(mapping, function(x) x$stage.mapped); is.na(st) <- lengths(st) == 0
  cm <- sapply(mapping, function(x) x$cells.mapped[1]); is.na(cm) <- lengths(cm) == 0
  out$mapping <- data.frame(
      cell            = names(mapping), 
      celltype.mapped = unlist(ct),
      stage.mapped    = unlist(st),
      closest.cell    = unlist(cm))
  
  out$mapping <- cbind(out$mapping,multinomial.prob)
  out$pca <- big_pca
  message("Done\n")
  
  return(out)


Computing mapping scores...

Done


Writing output...

Done




In [62]:
str(out)

List of 7
 $ closest.cells      :List of 25
  ..$ : Named chr [1:1000] "cell_393" "cell_965" "cell_1108" "cell_393" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr [1:1000] "cell_592" "cell_711" "cell_1386" "cell_1434" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr [1:1000] "cell_398" "cell_1099" "cell_1196" "cell_2671" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr